In [1]:
!nvidia-smi

Tue Oct  6 06:00:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U ultralytics nvidia-modelopt onnx onnxruntime onnxslim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 20.8 MB/s eta 0:00:00


In [3]:
import torch
import ultralytics
import modelopt.torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Ultralytics:", ultralytics.__version__)
print("ModelOpt:", modelopt.torch.__version__ if hasattr(modelopt.torch, "__version__") else "installed")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Ultralytics: 8.4.173
ModelOpt: installed


/usr/local/lib/python3.13/dist-packages/modelopt/torch/__init__.py:55: UserWarning: transformers 5.18.0 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


In [4]:
import torch
import modelopt.torch
from ultralytics.utils import torch_utils

print("=== Environment ===")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print()

print("=== ModelOpt QAT ===")

# Check that the important ModelOpt QAT components are available
qat_modules = [
    "modelopt.torch.quantization",
    "modelopt.torch.opt",
]

for module_name in qat_modules:
    try:
        __import__(module_name)
        print(f"✓ {module_name}")
    except Exception as e:
        print(f"✗ {module_name}: {e}")

# Check Ultralytics QAT utilities
for name in ["prepare_qat", "is_qat", "restore_qat"]:
    print(
        f"{'✓' if hasattr(torch_utils, name) else '✗'} "
        f"torch_utils.{name}"
    )

=== Environment ===
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4

=== ModelOpt QAT ===
✓ modelopt.torch.quantization
✓ modelopt.torch.opt
✓ torch_utils.prepare_qat
✓ torch_utils.is_qat
✓ torch_utils.restore_qat


In [5]:
import os
import glob
import zipfile
from pathlib import Path

from ultralytics import YOLO

# ============================================================
# CONFIG
# ============================================================

WORK_DIR = Path("/content/pcb_qat")
DATASET_DIR = WORK_DIR / "dataset"

WORK_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. FIND UPLOADED ZIP
# ============================================================

zip_candidates = glob.glob("/content/*.zip")

print("ZIP files found:")
for z in zip_candidates:
    print("  ", z)

pcb_zips = [
    z for z in zip_candidates
    if "pcb" in os.path.basename(z).lower()
]

if not pcb_zips:
    raise FileNotFoundError(
        "Could not find a PCB dataset ZIP in /content/"
    )

ZIP_PATH = pcb_zips[0]

print("\nUsing dataset:")
print(ZIP_PATH)

# ============================================================
# 2. EXTRACT DATASET
# ============================================================

if not DATASET_DIR.exists() or not any(DATASET_DIR.iterdir()):

    print("\nExtracting dataset...")

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATASET_DIR)

    print("Extraction complete.")

else:
    print("\nDataset already extracted. Skipping extraction.")

# ============================================================
# 3. FIND data.yaml
# ============================================================

yaml_candidates = list(DATASET_DIR.rglob("data.yaml"))

if not yaml_candidates:
    raise FileNotFoundError(
        "data.yaml was not found inside the extracted dataset."
    )

DATA_YAML = yaml_candidates[0]

print("\nDataset YAML:")
print(DATA_YAML)

# ============================================================
# 4. FIND ORIGINAL FP32 MODEL
# ============================================================

pt_candidates = glob.glob("/content/*.pt")

print("\nPT files found:")
for p in pt_candidates:
    print("  ", p)

preferred = [
    p for p in pt_candidates
    if "best" in os.path.basename(p).lower()
]

if not preferred:
    raise FileNotFoundError(
        "Could not find best .pt model in /content/"
    )

MODEL_PATH = preferred[0]

print("\nUsing FP32 model:")
print(MODEL_PATH)

# ============================================================
# 5. LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

print("\nModel loaded successfully.")
print("Model:", MODEL_PATH)

# ============================================================
# 6. FP32 BASELINE VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("RUNNING FP32 BASELINE")
print("=" * 60)

results = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    verbose=True
)

# ============================================================
# 7. PRINT BASELINE METRICS
# ============================================================

print("\n" + "=" * 60)
print("FP32 BASELINE RESULTS")
print("=" * 60)

print(f"mAP50:     {results.box.map50:.6f}")
print(f"mAP50-95:  {results.box.map:.6f}")
print(f"Precision: {results.box.mp:.6f}")
print(f"Recall:    {results.box.mr:.6f}")

print("\nFP32 baseline completed successfully.")

ZIP files found:
   /content/pcb-2.zip

Using dataset:
/content/pcb-2.zip

Extracting dataset...
Extraction complete.

Dataset YAML:
/content/pcb_qat/dataset/content/pcb-2/data.yaml

PT files found:
   /content/best (1).pt

Using FP32 model:
/content/best (1).pt

Model loaded successfully.
Model: /content/best (1).pt

RUNNING FP32 BASELINE
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1404.8±448.3 MB/s, size: 60.9 KB)
val: Scanning /content/pcb_qat/dataset/content/pcb-2/valid/labels... 798 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 798/798 977.5it/s 0.8s
val: New cache created: /content/pcb_qat/dataset/content/pcb-2/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 3.6it/s 28.0s
                   all        798       1588    

In [6]:
from ultralytics import YOLO
from pathlib import Path
import torch

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = "/content/best (1).pt"
DATA_YAML = "/content/pcb_qat/dataset/content/pcb-2/data.yaml"

OUTPUT_DIR = "/content/qat_output"

# ============================================================
# GPU CHECK
# ============================================================

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

# ============================================================
# LOAD ORIGINAL FP32 MODEL
# ============================================================

model = YOLO(MODEL_PATH)

# ============================================================
# INT8 QAT TRAINING
# ============================================================

results = model.train(
    data=DATA_YAML,

    # -----------------------------
    # QAT
    # -----------------------------
    quantize=8,

    # -----------------------------
    # Training
    # -----------------------------
    epochs=10,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,

    # VERY IMPORTANT FOR QAT
    compile=False,

    # Conservative fine-tuning
    optimizer="AdamW",
    lr0=1e-5,
    lrf=0.1,
    weight_decay=0.0005,
    patience=5,

    # -----------------------------
    # Augmentation
    # -----------------------------
    degrees=0.0,
    translate=0.05,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,

    mosaic=0.0,
    mixup=0.0,

    # -----------------------------
    # Validation
    # -----------------------------
    val=True,

    # -----------------------------
    # Reproducibility
    # -----------------------------
    seed=42,
    deterministic=True,

    # -----------------------------
    # Save
    # -----------------------------
    save=True,
    save_period=1,

    project=OUTPUT_DIR,
    name="yolov8m_qat",

    exist_ok=False
)

print("\nQAT training finished.")

GPU: Tesla T4
CUDA: True
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pcb_qat/dataset/content/pcb-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/best (1).pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov8

/usr/local/lib/python3.13/dist-packages/modelopt/torch/quantization/nn/modules/quant_module.py:111: UserWarning: Could not identify the device for TensorQuantizer states of model.9.m. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


Model summary (fused): 261 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.1it/s 16.3s
                   all        798       1588      0.981       0.99       0.99      0.622
          missing_hole        118        229      0.998      0.996      0.995      0.658
            mouse_bite        140        280      0.979      0.989      0.986      0.629
          open_circuit        133        255      0.984      0.993      0.991      0.602
                 short        156        324      0.978      0.975      0.987      0.633
                  spur        130        262      0.969      0.992      0.993      0.607
       spurious_copper        121        238      0.979      0.996      0.986      0.605
Speed: 0.5ms preprocess, 15.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/qat_output/yolov8m_qat
Inserted 408 quantizers

QAT train

/usr/local/lib/python3.13/dist-packages/modelopt/torch/quantization/nn/modules/quant_module.py:111: UserWarning: Could not identify the device for TensorQuantizer states of model.9.m. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


In [7]:
import torch
from ultralytics import YOLO

QAT_MODEL = "/content/qat_output/yolov8m_qat/weights/best.pt"

print("=" * 70)
print("LOADING QAT CHECKPOINT")
print("=" * 70)

model = YOLO(QAT_MODEL)

# Explicitly move to GPU to address the TensorQuantizer device warning
model.model = model.model.to("cuda:0")

# Count quantizer modules
quantizers = []

for name, module in model.model.named_modules():
    if "TensorQuantizer" in type(module).__name__:
        quantizers.append((name, type(module).__name__))

print(f"\nQAT checkpoint: {QAT_MODEL}")
print(f"Quantizer count: {len(quantizers)}")

print("\nFirst 10 quantizers:")
for name, qtype in quantizers[:10]:
    print(f"  {name} -> {qtype}")

# Use Ultralytics' official QAT detector
try:
    from ultralytics.utils.torch_utils import is_qat
    qat_status = is_qat(model.model)
except Exception as e:
    qat_status = f"CHECK_FAILED: {e}"

print("\nUltralytics is_qat():", qat_status)

print("\n" + "=" * 70)

if len(quantizers) == 408 and qat_status is True:
    print("✅ GENUINE QAT CHECKPOINT CONFIRMED")
    print("✅ 408 TensorQuantizer modules detected")
    print("✅ Safe to proceed to QAT ONNX export")
else:
    print("❌ QAT VERIFICATION FAILED")
    print("DO NOT EXPORT YET.")

LOADING QAT CHECKPOINT
Inserted 408 quantizers

QAT checkpoint: /content/qat_output/yolov8m_qat/weights/best.pt
Quantizer count: 408

First 10 quantizers:
  model.0.conv.input_quantizer -> TensorQuantizer
  model.0.conv.output_quantizer -> TensorQuantizer
  model.0.conv.weight_quantizer -> TensorQuantizer
  model.0.bn.input_quantizer -> TensorQuantizer
  model.0.bn.output_quantizer -> TensorQuantizer
  model.1.conv.input_quantizer -> TensorQuantizer
  model.1.conv.output_quantizer -> TensorQuantizer
  model.1.conv.weight_quantizer -> TensorQuantizer
  model.1.bn.input_quantizer -> TensorQuantizer
  model.1.bn.output_quantizer -> TensorQuantizer

Ultralytics is_qat(): True

✅ GENUINE QAT CHECKPOINT CONFIRMED
✅ 408 TensorQuantizer modules detected
✅ Safe to proceed to QAT ONNX export


In [8]:
from ultralytics import YOLO
from pathlib import Path
import torch

QAT_MODEL = "/content/qat_output/yolov8m_qat/weights/best.pt"

print("=" * 70)
print("QAT → INT8 ONNX EXPORT")
print("=" * 70)

model = YOLO(QAT_MODEL)

# Explicitly move model to CUDA
model.model = model.model.to("cuda:0")

# Verify QAT one more time
from ultralytics.utils.torch_utils import is_qat

print("QAT status:", is_qat(model.model))

if not is_qat(model.model):
    raise RuntimeError(
        "QAT checkpoint was not recognized as QAT. "
        "Export aborted."
    )

quantizer_count = sum(
    1
    for module in model.model.modules()
    if "TensorQuantizer" in type(module).__name__
)

print("TensorQuantizer count:", quantizer_count)

if quantizer_count != 408:
    raise RuntimeError(
        f"Expected 408 TensorQuantizers, found {quantizer_count}. "
        "Export aborted."
    )

print("\nExporting QAT checkpoint to ONNX...")
print("No calibration dataset will be supplied.")

exported = model.export(
    format="onnx",
    imgsz=640,
    opset=18,
    simplify=True,
    dynamic=False,
    half=False,
    device=0,
    compile=False
)

print("\n" + "=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print("Exported model:")
print(exported)

onnx_path = Path(exported)

if not onnx_path.exists():
    raise FileNotFoundError(
        f"ONNX file was not found: {onnx_path}"
    )

print(f"\nONNX size: {onnx_path.stat().st_size / (1024**2):.2f} MB")

QAT → INT8 ONNX EXPORT
Inserted 408 quantizers
QAT status: True
TensorQuantizer count: 408

Exporting QAT checkpoint to ONNX...
No calibration dataset will be supplied.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 261 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/content/qat_output/yolov8m_qat/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (49.9 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export success ✅ 10.1s, saved as '/content/qat_output/yolov8m_qat/weights/best.onnx' (99.0 MB)

Export complete (11.2s)
Results saved to /content/qat_output/yolov8m_qat/weights/b

In [9]:
import onnx
from collections import Counter

ONNX_PATH = "/content/qat_output/yolov8m_qat/weights/best.onnx"

print("=" * 70)
print("ONNX QUANTIZATION GRAPH ANALYSIS")
print("=" * 70)

model = onnx.load(ONNX_PATH)

# ------------------------------------------------------------
# 1. Basic information
# ------------------------------------------------------------

print(f"\nIR version: {model.ir_version}")
print(f"Opset: {model.opset_import[0].version}")

print(f"Inputs: {len(model.graph.input)}")
print(f"Outputs: {len(model.graph.output)}")
print(f"Nodes: {len(model.graph.node)}")
print(f"Initializers: {len(model.graph.initializer)}")

# ------------------------------------------------------------
# 2. Operator statistics
# ------------------------------------------------------------

ops = Counter(node.op_type for node in model.graph.node)

print("\n" + "=" * 70)
print("OPERATOR COUNTS")
print("=" * 70)

for op, count in ops.most_common():
    print(f"{op:30s} {count}")

# ------------------------------------------------------------
# 3. Q/DQ nodes
# ------------------------------------------------------------

quantize_nodes = [
    node for node in model.graph.node
    if node.op_type == "QuantizeLinear"
]

dequantize_nodes = [
    node for node in model.graph.node
    if node.op_type == "DequantizeLinear"
]

print("\n" + "=" * 70)
print("QUANTIZATION NODES")
print("=" * 70)

print("QuantizeLinear:   ", len(quantize_nodes))
print("DequantizeLinear: ", len(dequantize_nodes))

# ------------------------------------------------------------
# 4. Tensor data types
# ------------------------------------------------------------

dtype_names = {
    1: "FLOAT32",
    2: "UINT8",
    3: "INT8",
    4: "UINT16",
    5: "INT16",
    6: "INT32",
    7: "INT64",
    9: "BOOL",
    10: "FLOAT16",
    11: "DOUBLE",
}

initializer_types = Counter()

for tensor in model.graph.initializer:
    initializer_types[dtype_names.get(tensor.data_type, str(tensor.data_type))] += 1

print("\n" + "=" * 70)
print("INITIALIZER DATA TYPES")
print("=" * 70)

for dtype, count in initializer_types.items():
    print(f"{dtype:15s} {count}")

# ------------------------------------------------------------
# 5. Quantization parameter data types
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE Q/DQ NODES")
print("=" * 70)

for node in quantize_nodes[:10]:
    print("QuantizeLinear:", node.name)
    print("  inputs :", list(node.input))
    print("  outputs:", list(node.output))

for node in dequantize_nodes[:10]:
    print("DequantizeLinear:", node.name)
    print("  inputs :", list(node.input))
    print("  outputs:", list(node.output))

# ------------------------------------------------------------
# 6. Final conclusion
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INITIAL ASSESSMENT")
print("=" * 70)

if len(quantize_nodes) > 0 and len(dequantize_nodes) > 0:
    print("✅ Q/DQ quantization structure detected.")
else:
    print("❌ No Q/DQ structure detected.")

if initializer_types.get("INT8", 0) > 0 or initializer_types.get("UINT8", 0) > 0:
    print("✅ INT8/UINT8 initializers detected.")
else:
    print("⚠️ No INT8/UINT8 initializers detected.")

if initializer_types.get("FLOAT32", 0) > 0:
    print("ℹ️ FLOAT32 initializers are present.")
    print("   This does NOT automatically mean the model is not INT8.")
    print("   We need to determine whether those tensors belong to")
    print("   intentionally floating-point layers.")

ONNX QUANTIZATION GRAPH ANALYSIS

IR version: 8
Opset: 18
Inputs: 1
Outputs: 1
Nodes: 603
Initializers: 339

OPERATOR COUNTS
QuantizeLinear                 152
DequantizeLinear               152
Conv                           84
Sigmoid                        78
Mul                            78
Concat                         17
Add                            14
Split                          8
Reshape                        8
MaxPool                        3
Resize                         2
Slice                          2
Sub                            2
Transpose                      1
Softmax                        1
Div                            1

QUANTIZATION NODES
QuantizeLinear:    152
DequantizeLinear:  152

INITIALIZER DATA TYPES
FLOAT32         319
INT8            8
INT64           12

SAMPLE Q/DQ NODES
QuantizeLinear: /model.0/conv/input_quantizer/QuantizeLinear
  inputs : ['images', '/model.0/conv/input_quantizer/Constant_1_output_0', '/model.0/conv/input_quantizer/Const

In [10]:
import onnx
from collections import defaultdict

ONNX_PATH = "/content/qat_output/yolov8m_qat/weights/best.onnx"

model = onnx.load(ONNX_PATH)

# ONNX tensor type mapping
DTYPES = {
    1: "FLOAT32",
    2: "UINT8",
    3: "INT8",
    4: "UINT16",
    5: "INT16",
    6: "INT32",
    7: "INT64",
    9: "BOOL",
    10: "FLOAT16",
    11: "DOUBLE",
}

# Collect value_info for tensors
tensor_types = {}

for value in list(model.graph.value_info) + \
              list(model.graph.input) + \
              list(model.graph.output):

    try:
        elem_type = value.type.tensor_type.elem_type
        tensor_types[value.name] = DTYPES.get(
            elem_type,
            f"TYPE_{elem_type}"
        )
    except:
        pass

# Initializers
for init in model.graph.initializer:
    tensor_types[init.name] = DTYPES.get(
        init.data_type,
        f"TYPE_{init.data_type}"
    )

print("=" * 70)
print("CONVOLUTION TENSOR TYPE ANALYSIS")
print("=" * 70)

conv_count = 0
int8_related = 0
float_related = 0

for node in model.graph.node:

    if node.op_type != "Conv":
        continue

    conv_count += 1

    input_types = [
        tensor_types.get(x, "UNKNOWN")
        for x in node.input
    ]

    output_types = [
        tensor_types.get(x, "UNKNOWN")
        for x in node.output
    ]

    print(f"\nConv #{conv_count}")
    print("Name:", node.name)
    print("Inputs :", input_types)
    print("Outputs:", output_types)

    if "INT8" in input_types or "INT8" in output_types:
        int8_related += 1

    if "FLOAT32" in input_types or "FLOAT32" in output_types:
        float_related += 1

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print("Total Conv operators:", conv_count)
print("INT8-related Conv:", int8_related)
print("FLOAT32-related Conv:", float_related)

print("\nNote:")
print("UNKNOWN means ONNX did not preserve explicit tensor type")
print("information in graph.value_info for that tensor.")

CONVOLUTION TENSOR TYPE ANALYSIS

Conv #1
Name: /model.0/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #2
Name: /model.1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #3
Name: /model.2/cv1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #4
Name: /model.2/m.0/cv1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #5
Name: /model.2/m.0/cv2/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #6
Name: /model.2/m.1/cv1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #7
Name: /model.2/m.1/cv2/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #8
Name: /model.2/cv2/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #9
Name: /model.3/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #10
Name: /model.4/cv1/conv/Conv
Inputs

In [11]:
# ============================================================
# YOLOv8m QAT -> INT8 ONNX
# Correct QAT export
# ============================================================

from ultralytics import YOLO
from ultralytics.utils.torch_utils import is_qat
from pathlib import Path
import torch
import onnx
import os

# ------------------------------------------------------------
# 1. PATH
# ------------------------------------------------------------

QAT_MODEL = "/content/qat_output/yolov8m_qat/weights/best.pt"

IMG_SIZE = 640

print("=" * 70)
print("YOLOv8m QAT -> INT8 ONNX EXPORT")
print("=" * 70)

# ------------------------------------------------------------
# 2. GPU CHECK
# ------------------------------------------------------------

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

# ------------------------------------------------------------
# 3. LOAD QAT CHECKPOINT
# ------------------------------------------------------------

print("\nLoading QAT checkpoint...")

model = YOLO(QAT_MODEL)

# Move to GPU
model.model = model.model.to("cuda:0")

# ------------------------------------------------------------
# 4. VERIFY QAT
# ------------------------------------------------------------

qat_status = is_qat(model.model)

quantizer_count = sum(
    1
    for module in model.model.modules()
    if "TensorQuantizer" in type(module).__name__
)

print("\nQAT status:", qat_status)
print("TensorQuantizer count:", quantizer_count)

if not qat_status:
    raise RuntimeError(
        "ERROR: checkpoint is not recognized as a QAT model."
    )

if quantizer_count != 408:
    raise RuntimeError(
        f"ERROR: expected 408 TensorQuantizers, "
        f"found {quantizer_count}."
    )

print("\n✅ Genuine QAT checkpoint confirmed.")
print("✅ 408 TensorQuantizers detected.")

# ------------------------------------------------------------
# 5. EXPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPORTING QAT CHECKPOINT")
print("=" * 70)

print("\nUsing:")
print("  format   = onnx")
print("  quantize = 8")
print("  imgsz    = 640")
print("  opset    = 18")
print("  dynamic  = False")
print("  simplify = True")

exported_path = model.export(
    format="onnx",
    quantize=8,          # IMPORTANT
    imgsz=IMG_SIZE,
    opset=18,
    simplify=True,
    dynamic=False,
    device=0,
    compile=False
)

print("\nExport returned:")
print(exported_path)

# ------------------------------------------------------------
# 6. CHECK FILE
# ------------------------------------------------------------

ONNX_PATH = Path(exported_path)

if not ONNX_PATH.exists():
    raise FileNotFoundError(
        f"ONNX file was not created:\n{ONNX_PATH}"
    )

size_mb = ONNX_PATH.stat().st_size / (1024 ** 2)

print("\n" + "=" * 70)
print("EXPORT SUCCESS")
print("=" * 70)

print("\nONNX path:")
print(ONNX_PATH)

print(f"\nONNX size: {size_mb:.2f} MB")

# ------------------------------------------------------------
# 7. ONNX VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING ONNX GRAPH")
print("=" * 70)

onnx_model = onnx.load(str(ONNX_PATH))

onnx.checker.check_model(onnx_model)

print("ONNX checker: ✅ PASS")

# ------------------------------------------------------------
# 8. COUNT Q/DQ
# ------------------------------------------------------------

quantize_nodes = [
    node
    for node in onnx_model.graph.node
    if node.op_type == "QuantizeLinear"
]

dequantize_nodes = [
    node
    for node in onnx_model.graph.node
    if node.op_type == "DequantizeLinear"
]

print("\nONNX nodes:", len(onnx_model.graph.node))
print("QuantizeLinear:", len(quantize_nodes))
print("DequantizeLinear:", len(dequantize_nodes))

# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

print("""
✅ QAT checkpoint loaded
✅ QAT verified
✅ 408 TensorQuantizers verified
✅ quantize=8 used during ONNX export
✅ ONNX graph validated
""")

print("IMPORTANT:")
print("This is an INT8 QAT ONNX/QDQ model.")
print("We will NOT claim it is 100% integer-only until")
print("we inspect the actual operator/tensor types.")

YOLOv8m QAT -> INT8 ONNX EXPORT

GPU: Tesla T4
CUDA: True

Loading QAT checkpoint...
Inserted 408 quantizers

QAT status: True
TensorQuantizer count: 408

✅ Genuine QAT checkpoint confirmed.
✅ 408 TensorQuantizers detected.

EXPORTING QAT CHECKPOINT

Using:
  format   = onnx
  quantize = 8
  imgsz    = 640
  opset    = 18
  dynamic  = False
  simplify = True
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 261 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/content/qat_output/yolov8m_qat/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (49.9 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export success ✅ 10.5s, saved as '/content/qat_output/yolov8m_qat/weights/best.onnx' (99.0 MB)

Export complete (12.2s)
Results saved to /content/qat_output/yolov8m_qat/weights/best.onnx
Predict:         yolo predict t

In [12]:
# ============================================================
# FINAL INT8 GRAPH ANALYSIS
# ============================================================

import onnx
from collections import Counter

ONNX_PATH = "/content/qat_output/yolov8m_qat/weights/best.onnx"

model = onnx.load(ONNX_PATH)

DTYPES = {
    1: "FLOAT32",
    2: "UINT8",
    3: "INT8",
    4: "UINT16",
    5: "INT16",
    6: "INT32",
    7: "INT64",
    9: "BOOL",
    10: "FLOAT16",
    11: "DOUBLE",
}

# ------------------------------------------------------------
# Collect explicit tensor types
# ------------------------------------------------------------

tensor_types = {}

for value in (
    list(model.graph.value_info)
    + list(model.graph.input)
    + list(model.graph.output)
):
    try:
        tensor_types[value.name] = DTYPES.get(
            value.type.tensor_type.elem_type,
            f"TYPE_{value.type.tensor_type.elem_type}"
        )
    except Exception:
        pass

for init in model.graph.initializer:
    tensor_types[init.name] = DTYPES.get(
        init.data_type,
        f"TYPE_{init.data_type}"
    )

# ------------------------------------------------------------
# Operator counts
# ------------------------------------------------------------

ops = Counter(node.op_type for node in model.graph.node)

print("=" * 70)
print("FINAL INT8/QAT ONNX GRAPH ANALYSIS")
print("=" * 70)

print("\nTotal nodes:", len(model.graph.node))

print("\nOperator counts:")
for op, count in ops.most_common():
    print(f"{op:25s} {count}")

# ------------------------------------------------------------
# Q/DQ
# ------------------------------------------------------------

q_nodes = [
    n for n in model.graph.node
    if n.op_type == "QuantizeLinear"
]

dq_nodes = [
    n for n in model.graph.node
    if n.op_type == "DequantizeLinear"
]

print("\n" + "=" * 70)
print("QUANTIZATION")
print("=" * 70)

print("QuantizeLinear:", len(q_nodes))
print("DequantizeLinear:", len(dq_nodes))

# ------------------------------------------------------------
# Conv analysis
# ------------------------------------------------------------

conv_nodes = [
    n for n in model.graph.node
    if n.op_type == "Conv"
]

print("\n" + "=" * 70)
print("CONVOLUTION ANALYSIS")
print("=" * 70)

int8_conv = 0
float_conv = 0
unknown_conv = 0

for i, node in enumerate(conv_nodes, 1):

    input_types = [
        tensor_types.get(x, "UNKNOWN")
        for x in node.input
    ]

    output_types = [
        tensor_types.get(x, "UNKNOWN")
        for x in node.output
    ]

    print(f"\nConv #{i}")
    print("Name:", node.name)
    print("Inputs :", input_types)
    print("Outputs:", output_types)

    if (
        "INT8" in input_types
        or "INT8" in output_types
    ):
        int8_conv += 1

    elif (
        "FLOAT32" in input_types
        or "FLOAT32" in output_types
    ):
        float_conv += 1

    else:
        unknown_conv += 1

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print("Total Conv:", len(conv_nodes))
print("INT8-related Conv:", int8_conv)
print("FLOAT32-related Conv:", float_conv)
print("Unknown Conv:", unknown_conv)

# ------------------------------------------------------------
# Initializer data types
# ------------------------------------------------------------

initializer_types = Counter()

for init in model.graph.initializer:
    dtype = DTYPES.get(
        init.data_type,
        f"TYPE_{init.data_type}"
    )
    initializer_types[dtype] += 1

print("\n" + "=" * 70)
print("INITIALIZER TYPES")
print("=" * 70)

for dtype, count in initializer_types.items():
    print(f"{dtype:15s}: {count}")

# ------------------------------------------------------------
# FINAL INTERPRETATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

if int8_conv == len(conv_nodes):
    print("✅ Conv operators are INT8-related.")
else:
    print(
        "⚠️ Conv operators are not represented as native INT8 "
        "operators in the ONNX graph."
    )

if len(q_nodes) > 0 and len(dq_nodes) > 0:
    print("✅ Q/DQ quantization structure exists.")

print("\nIMPORTANT:")
print("QAT + Q/DQ does not automatically mean every ONNX")
print("operator is natively INT8.")
print("The Conv tensor types determine that distinction.")

FINAL INT8/QAT ONNX GRAPH ANALYSIS

Total nodes: 603

Operator counts:
QuantizeLinear            152
DequantizeLinear          152
Conv                      84
Sigmoid                   78
Mul                       78
Concat                    17
Add                       14
Split                     8
Reshape                   8
MaxPool                   3
Resize                    2
Slice                     2
Sub                       2
Transpose                 1
Softmax                   1
Div                       1

QUANTIZATION
QuantizeLinear: 152
DequantizeLinear: 152

CONVOLUTION ANALYSIS

Conv #1
Name: /model.0/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #2
Name: /model.1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #3
Name: /model.2/cv1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['FLOAT32']

Conv #4
Name: /model.2/m.0/cv1/conv/Conv
Inputs : ['FLOAT32', 'FLOAT32', 'FLOAT32']
Outputs: ['

In [13]:
import onnx
import os
from collections import Counter

PTQ_ONNX = "/content/best (1)_int8.onnx"

model = onnx.load(PTQ_ONNX)

print("=" * 70)
print("OLD 25 MB PTQ INT8 MODEL ANALYSIS")
print("=" * 70)

print("\nFile size:")
print(f"{os.path.getsize(PTQ_ONNX) / (1024 * 1024):.2f} MB")

# ------------------------------------------------------------
# Operator counts
# ------------------------------------------------------------

ops = Counter(node.op_type for node in model.graph.node)

print("\nOperator counts:")

for op, count in ops.most_common():
    print(f"{op:<25} {count}")

# ------------------------------------------------------------
# Tensor type mapping
# ------------------------------------------------------------

tensor_types = {}

for initializer in model.graph.initializer:
    tensor_types[initializer.name] = initializer.data_type

for value_info in (
    list(model.graph.input)
    + list(model.graph.output)
    + list(model.graph.value_info)
):
    tensor_types[value_info.name] = value_info.type.tensor_type.elem_type

TYPE_MAP = {
    1: "FLOAT32",
    2: "UINT8",
    3: "INT8",
    6: "INT32",
    7: "INT64",
    10: "FLOAT16",
}

# ------------------------------------------------------------
# Initializer types
# ------------------------------------------------------------

initializer_counts = Counter(
    TYPE_MAP.get(x.data_type, f"TYPE_{x.data_type}")
    for x in model.graph.initializer
)

print("\nInitializer types:")

for dtype, count in initializer_counts.items():
    print(f"{dtype:<15} {count}")

# ------------------------------------------------------------
# Conv analysis
# ------------------------------------------------------------

conv_count = 0
int8_conv = 0
float_conv = 0

print("\n" + "=" * 70)
print("CONVOLUTION ANALYSIS")
print("=" * 70)

for node in model.graph.node:

    if node.op_type != "Conv":
        continue

    conv_count += 1

    input_types = [
        TYPE_MAP.get(tensor_types.get(name), "UNKNOWN")
        for name in node.input
    ]

    output_types = [
        TYPE_MAP.get(tensor_types.get(name), "UNKNOWN")
        for name in node.output
    ]

    print(f"\nConv #{conv_count}")
    print("Name:", node.name)
    print("Inputs :", input_types)
    print("Outputs:", output_types)

    if "INT8" in input_types or "INT8" in output_types:
        int8_conv += 1

    if "FLOAT32" in input_types or "FLOAT32" in output_types:
        float_conv += 1

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print("Total Conv:", conv_count)
print("INT8-related Conv:", int8_conv)
print("FLOAT32-related Conv:", float_conv)

print("\nQ/DQ:")
print("QuantizeLinear :", ops.get("QuantizeLinear", 0))
print("DequantizeLinear:", ops.get("DequantizeLinear", 0))

print("\nQLinearConv:", ops.get("QLinearConv", 0))

print("\nAnalysis complete.")

OLD 25 MB PTQ INT8 MODEL ANALYSIS

File size:
25.10 MB

Operator counts:
DequantizeLinear          330
QuantizeLinear            163
Conv                      84
Sigmoid                   78
Mul                       78
Concat                    17
Add                       14
Split                     8
Reshape                   8
MaxPool                   3
Resize                    2
Slice                     2
Sub                       2
Transpose                 1
Softmax                   1
Div                       1

Initializer types:
INT64           12
FLOAT32         335
INT8            331
INT32           166

CONVOLUTION ANALYSIS

Conv #1
Name: /model.0/conv/Conv
Inputs : ['UNKNOWN', 'UNKNOWN', 'UNKNOWN']
Outputs: ['FLOAT32']

Conv #2
Name: /model.1/conv/Conv
Inputs : ['UNKNOWN', 'UNKNOWN', 'UNKNOWN']
Outputs: ['FLOAT32']

Conv #3
Name: /model.2/cv1/conv/Conv
Inputs : ['UNKNOWN', 'UNKNOWN', 'UNKNOWN']
Outputs: ['FLOAT32']

Conv #4
Name: /model.2/m.0/cv1/conv/Conv
Inputs : 

In [14]:
import onnx
import numpy as np
from onnx import numpy_helper

QAT_ONNX = "/content/qat_output/yolov8m_qat/weights/best.onnx"

model = onnx.load(QAT_ONNX)
graph = model.graph

# ------------------------------------------------------------
# Build lookup tables
# ------------------------------------------------------------

initializer_map = {
    init.name: init
    for init in graph.initializer
}

node_by_output = {}

for node in graph.node:
    for output in node.output:
        node_by_output[output] = node

def dtype_name(onnx_dtype):
    mapping = {
        onnx.TensorProto.FLOAT: "FLOAT32",
        onnx.TensorProto.INT8: "INT8",
        onnx.TensorProto.UINT8: "UINT8",
        onnx.TensorProto.INT32: "INT32",
        onnx.TensorProto.INT64: "INT64",
        onnx.TensorProto.FLOAT16: "FLOAT16",
    }
    return mapping.get(onnx_dtype, f"TYPE_{onnx_dtype}")

# ------------------------------------------------------------
# Find first 10 Conv nodes
# ------------------------------------------------------------

conv_nodes = [
    node for node in graph.node
    if node.op_type == "Conv"
]

print("=" * 80)
print("QAT ONNX WEIGHT / ACTIVATION STRUCTURE")
print("=" * 80)

print(f"\nTotal Conv nodes: {len(conv_nodes)}")

for idx, conv in enumerate(conv_nodes[:10], start=1):

    print("\n" + "-" * 80)
    print(f"CONV #{idx}")
    print("-" * 80)

    print("Conv name:")
    print(conv.name)

    print("\nConv inputs:")
    for i, inp in enumerate(conv.input):
        print(f"  input[{i}]: {inp}")

        if inp in initializer_map:
            init = initializer_map[inp]
            print(
                f"      initializer dtype = {dtype_name(init.data_type)}"
            )
            print(
                f"      shape             = {list(init.dims)}"
            )

        producer = node_by_output.get(inp)

        if producer is not None:
            print(
                f"      producer          = "
                f"{producer.op_type} ({producer.name})"
            )

            if producer.op_type == "DequantizeLinear":
                print("      >>> THIS INPUT COMES FROM DQ <<<")

                print("\n      DQ inputs:")

                for j, dq_input in enumerate(producer.input):
                    print(f"        dq_input[{j}]: {dq_input}")

                    if dq_input in initializer_map:
                        dq_init = initializer_map[dq_input]

                        print(
                            f"            dtype = "
                            f"{dtype_name(dq_init.data_type)}"
                        )

                        arr = numpy_helper.to_array(dq_init)

                        print(
                            f"            shape = {arr.shape}"
                        )

                        if arr.size <= 20:
                            print(
                                f"            values = {arr}"
                            )
                        else:
                            print(
                                f"            min = {arr.min()}"
                            )
                            print(
                                f"            max = {arr.max()}"
                            )

    print("\nConv outputs:")
    for out in conv.output:
        print(f"  {out}")

        consumer_nodes = [
            node for node in graph.node
            if out in node.input
        ]

        for consumer in consumer_nodes[:3]:
            print(
                f"      consumer = "
                f"{consumer.op_type} ({consumer.name})"
            )

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

QAT ONNX WEIGHT / ACTIVATION STRUCTURE

Total Conv nodes: 84

--------------------------------------------------------------------------------
CONV #1
--------------------------------------------------------------------------------
Conv name:
/model.0/conv/Conv

Conv inputs:
  input[0]: /model.0/conv/input_quantizer/DequantizeLinear_output_0
      producer          = DequantizeLinear (/model.0/conv/input_quantizer/DequantizeLinear)
      >>> THIS INPUT COMES FROM DQ <<<

      DQ inputs:
        dq_input[0]: /model.0/conv/input_quantizer/QuantizeLinear_output_0
        dq_input[1]: /model.0/conv/input_quantizer/Constant_1_output_0
            dtype = FLOAT32
            shape = ()
            values = 0.007874015718698502
        dq_input[2]: /model.0/conv/input_quantizer/Constant_output_0
            dtype = INT8
            shape = ()
            values = 0
  input[1]: /model.0/conv/weight_quantizer/DequantizeLinear_output_0
      producer          = DequantizeLinear (/model.0/conv/w

In [15]:
# ============================================================
# QAT QDQ -> PACKED INT8 WEIGHTS
#
# Purpose:
#   Preserve QAT learned quantization parameters while
#   physically storing convolution weights as INT8.
#
# This is NOT another PTQ calibration pass.
# ============================================================

import onnx
import numpy as np
import os
from onnx import numpy_helper, TensorProto

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

QAT_ONNX = "/content/qat_output/yolov8m_qat/weights/best.onnx"

OUTPUT_ONNX = (
    "/content/qat_output/yolov8m_qat/weights/"
    "best_qat_fiq_packed.onnx"
)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

print("=" * 80)
print("QAT -> PACKED INT8 WEIGHT CONVERSION")
print("=" * 80)

print("\nLoading:")
print(QAT_ONNX)

model = onnx.load(QAT_ONNX)
graph = model.graph

# ------------------------------------------------------------
# Build maps
# ------------------------------------------------------------

initializers = {
    init.name: init
    for init in graph.initializer
}

producer_map = {}

for node in graph.node:
    for output in node.output:
        producer_map[output] = node

# ------------------------------------------------------------
# Find weight Q -> DQ patterns
#
# We identify a QuantizeLinear whose input is an initializer.
# That distinguishes weight quantization from activation
# quantization.
# ------------------------------------------------------------

weight_q_nodes = []
weight_dq_nodes = []

for dq_node in graph.node:

    if dq_node.op_type != "DequantizeLinear":
        continue

    if len(dq_node.input) < 3:
        continue

    q_output = dq_node.input[0]

    q_node = producer_map.get(q_output)

    if q_node is None:
        continue

    if q_node.op_type != "QuantizeLinear":
        continue

    # The first Q input is the tensor being quantized.
    source_name = q_node.input[0]

    # A weight is stored as an initializer.
    if source_name not in initializers:
        continue

    weight_q_nodes.append(q_node)
    weight_dq_nodes.append(dq_node)

print("\nWeight Q/DQ pairs found:", len(weight_q_nodes))

# ------------------------------------------------------------
# Convert each weight
# ------------------------------------------------------------

converted = 0
already_int8 = 0

for q_node, dq_node in zip(weight_q_nodes, weight_dq_nodes):

    weight_name = q_node.input[0]
    scale_name = q_node.input[1]
    zero_point_name = (
        q_node.input[2]
        if len(q_node.input) >= 3
        else None
    )

    weight_init = initializers[weight_name]

    weight_fp = numpy_helper.to_array(weight_init)

    # --------------------------------------------------------
    # Scale
    # --------------------------------------------------------

    scale_init = initializers.get(scale_name)

    if scale_init is None:
        print(
            f"\nWARNING: scale initializer not found for {weight_name}"
        )
        continue

    scale = numpy_helper.to_array(scale_init).astype(np.float32)

    # --------------------------------------------------------
    # Zero point
    # --------------------------------------------------------

    if zero_point_name is not None:

        zp_init = initializers.get(zero_point_name)

        if zp_init is not None:
            zero_point = numpy_helper.to_array(
                zp_init
            ).astype(np.int32)
        else:
            zero_point = np.array(
                0,
                dtype=np.int32
            )

    else:
        zero_point = np.array(
            0,
            dtype=np.int32
        )

    # --------------------------------------------------------
    # Already INT8?
    # --------------------------------------------------------

    if weight_init.data_type == TensorProto.INT8:

        already_int8 += 1

        # Remove QuantizeLinear because the weight is already
        # quantized.
        q_node.input[0] = weight_name

        continue

    # --------------------------------------------------------
    # Determine quantization axis
    #
    # ModelOpt normally uses per-channel weight quantization
    # with axis=0.
    # --------------------------------------------------------

    axis = 0

    for attr in q_node.attribute:

        if attr.name == "axis":
            axis = attr.i

    # Normalize negative axis
    if axis < 0:
        axis += weight_fp.ndim

    # --------------------------------------------------------
    # Reshape scale / zero point for broadcasting
    # --------------------------------------------------------

    if scale.size > 1:

        scale_shape = [1] * weight_fp.ndim
        scale_shape[axis] = scale.size

        scale_broadcast = scale.reshape(scale_shape)

    else:

        scale_broadcast = scale.reshape(
            [1] * weight_fp.ndim
        )

    if zero_point.size > 1:

        zp_shape = [1] * weight_fp.ndim
        zp_shape[axis] = zero_point.size

        zp_broadcast = zero_point.reshape(zp_shape)

    else:

        zp_broadcast = zero_point.reshape(
            [1] * weight_fp.ndim
        )

    # --------------------------------------------------------
    # Perform the SAME INT8 quantization represented by Q
    # --------------------------------------------------------

    q_weight = np.rint(
        weight_fp / scale_broadcast
        + zp_broadcast
    )

    # --------------------------------------------------------
    # Saturate to INT8
    # --------------------------------------------------------

    q_weight = np.clip(
        q_weight,
        -128,
        127
    ).astype(np.int8)

    # --------------------------------------------------------
    # Replace FP32 initializer with INT8 initializer
    # --------------------------------------------------------

    new_initializer = numpy_helper.from_array(
        q_weight,
        name=weight_name
    )

    # Replace initializer in graph
    for i, existing in enumerate(graph.initializer):

        if existing.name == weight_name:

            graph.initializer.remove(existing)
            graph.initializer.insert(
                i,
                new_initializer
            )

            break

    # --------------------------------------------------------
    # Remove the QuantizeLinear stage from the weight path
    #
    # Conv's weight input still points to the DQ output.
    # We simply make DQ consume the INT8 initializer directly.
    # --------------------------------------------------------

    dq_node.input[0] = weight_name

    converted += 1


print("\nConverted FP32 weight tensors:", converted)
print("Already INT8 weight tensors  :", already_int8)

# ------------------------------------------------------------
# Remove weight QuantizeLinear nodes
# ------------------------------------------------------------

weight_q_names = {
    node.name
    for node in weight_q_nodes
}

new_nodes = []

removed_q = 0

for node in graph.node:

    if node.name in weight_q_names:

        removed_q += 1

        continue

    new_nodes.append(node)

# Replace node list
del graph.node[:]
graph.node.extend(new_nodes)

print("\nRemoved weight QuantizeLinear nodes:", removed_q)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

onnx.save(model, OUTPUT_ONNX)

print("\nSaved:")
print(OUTPUT_ONNX)

# ------------------------------------------------------------
# File size
# ------------------------------------------------------------

size_mb = os.path.getsize(
    OUTPUT_ONNX
) / (1024 * 1024)

original_size_mb = os.path.getsize(
    QAT_ONNX
) / (1024 * 1024)

print("\n" + "=" * 80)
print("SIZE")
print("=" * 80)

print(
    f"Original QAT ONNX : {original_size_mb:.2f} MB"
)

print(
    f"Packed QAT ONNX   : {size_mb:.2f} MB"
)

print(
    f"Reduction         : "
    f"{original_size_mb - size_mb:.2f} MB"
)

# ------------------------------------------------------------
# ONNX CHECK
# ------------------------------------------------------------

print("\nRunning ONNX checker...")

check_model = onnx.load(OUTPUT_ONNX)

onnx.checker.check_model(check_model)

print("ONNX checker: PASS")

# ------------------------------------------------------------
# Final initializer statistics
# ------------------------------------------------------------

from collections import Counter

dtype_map = {
    TensorProto.FLOAT: "FLOAT32",
    TensorProto.FLOAT16: "FLOAT16",
    TensorProto.INT8: "INT8",
    TensorProto.UINT8: "UINT8",
    TensorProto.INT32: "INT32",
    TensorProto.INT64: "INT64",
}

counts = Counter(
    dtype_map.get(
        init.data_type,
        f"TYPE_{init.data_type}"
    )
    for init in check_model.graph.initializer
)

print("\nFinal initializer types:")

for dtype, count in counts.items():

    print(
        f"{dtype:<12}: {count}"
    )

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

QAT -> PACKED INT8 WEIGHT CONVERSION

Loading:
/content/qat_output/yolov8m_qat/weights/best.onnx

Weight Q/DQ pairs found: 77

Converted FP32 weight tensors: 77
Already INT8 weight tensors  : 0

Removed weight QuantizeLinear nodes: 77

Saved:
/content/qat_output/yolov8m_qat/weights/best_qat_fiq_packed.onnx

SIZE
Original QAT ONNX : 99.01 MB
Packed QAT ONNX   : 25.15 MB
Reduction         : 73.87 MB

Running ONNX checker...
ONNX checker: PASS

Final initializer types:
FLOAT32     : 242
INT8        : 85
INT64       : 12

DONE


In [16]:
import onnx
import onnxruntime as ort
import numpy as np
import os

QAT_ONNX = "/content/qat_output/yolov8m_qat/weights/best.onnx"
FIQ_ONNX = "/content/qat_output/yolov8m_qat/weights/best_qat_fiq_packed.onnx"

print("=" * 80)
print("QAT ONNX vs PACKED QAT-FIQ RUNTIME TEST")
print("=" * 80)

# ------------------------------------------------------------
# File sizes
# ------------------------------------------------------------

qat_size = os.path.getsize(QAT_ONNX) / (1024 * 1024)
fiq_size = os.path.getsize(FIQ_ONNX) / (1024 * 1024)

print(f"\nQAT ONNX size    : {qat_size:.2f} MB")
print(f"Packed ONNX size : {fiq_size:.2f} MB")
print(f"Reduction        : {qat_size - fiq_size:.2f} MB")

# ------------------------------------------------------------
# Load ONNX Runtime sessions
# ------------------------------------------------------------

print("\nLoading ONNX Runtime sessions...")

qat_session = ort.InferenceSession(
    QAT_ONNX,
    providers=["CPUExecutionProvider"]
)

fiq_session = ort.InferenceSession(
    FIQ_ONNX,
    providers=["CPUExecutionProvider"]
)

print("QAT providers:")
print(qat_session.get_providers())

print("\nPacked providers:")
print(fiq_session.get_providers())

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

qat_input = qat_session.get_inputs()[0]
fiq_input = fiq_session.get_inputs()[0]

print("\nQAT input:")
print(" name :", qat_input.name)
print(" shape:", qat_input.shape)
print(" type :", qat_input.type)

print("\nPacked input:")
print(" name :", fiq_input.name)
print(" shape:", fiq_input.shape)
print(" type :", fiq_input.type)

# ------------------------------------------------------------
# Create deterministic test input
#
# This is only a numerical sanity test.
# It is NOT validation accuracy.
# ------------------------------------------------------------

rng = np.random.default_rng(12345)

dummy = rng.random(
    (1, 3, 640, 640),
    dtype=np.float32
)

print("\nRunning QAT ONNX...")
qat_outputs = qat_session.run(
    None,
    {qat_input.name: dummy}
)

print("Running packed ONNX...")
fiq_outputs = fiq_session.run(
    None,
    {fiq_input.name: dummy}
)

# ------------------------------------------------------------
# Output structure
# ------------------------------------------------------------

print("\nNumber of outputs:")
print(" QAT   :", len(qat_outputs))
print(" Packed:", len(fiq_outputs))

# ------------------------------------------------------------
# Compare outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERICAL COMPARISON")
print("=" * 80)

for i, (a, b) in enumerate(zip(qat_outputs, fiq_outputs)):

    a = np.asarray(a)
    b = np.asarray(b)

    print(f"\nOutput #{i}")

    print("QAT shape   :", a.shape)
    print("Packed shape:", b.shape)

    if a.shape != b.shape:

        print("WARNING: shape mismatch")

        continue

    abs_diff = np.abs(a - b)

    max_diff = np.max(abs_diff)
    mean_diff = np.mean(abs_diff)

    denom = np.maximum(np.abs(a), 1e-8)

    relative_diff = np.abs(a - b) / denom

    max_relative = np.max(relative_diff)
    mean_relative = np.mean(relative_diff)

    print(f"Max absolute difference : {max_diff:.8f}")
    print(f"Mean absolute difference: {mean_diff:.8f}")
    print(f"Max relative difference : {max_relative:.8f}")
    print(f"Mean relative difference: {mean_relative:.8f}")

print("\n" + "=" * 80)
print("RUNTIME TEST COMPLETE")
print("=" * 80)

QAT ONNX vs PACKED QAT-FIQ RUNTIME TEST

QAT ONNX size    : 99.01 MB
Packed ONNX size : 25.15 MB
Reduction        : 73.87 MB

Loading ONNX Runtime sessions...
QAT providers:
['CPUExecutionProvider']

Packed providers:
['CPUExecutionProvider']

QAT input:
 name : images
 shape: [1, 3, 640, 640]
 type : tensor(float)

Packed input:
 name : images
 shape: [1, 3, 640, 640]
 type : tensor(float)

Running QAT ONNX...
Running packed ONNX...

Number of outputs:
 QAT   : 1
 Packed: 1

NUMERICAL COMPARISON

Output #0
QAT shape   : (1, 10, 8400)
Packed shape: (1, 10, 8400)
Max absolute difference : 0.00000000
Mean absolute difference: 0.00000000
Max relative difference : 0.00000000
Mean relative difference: 0.00000000

RUNTIME TEST COMPLETE


In [17]:
import onnx
import onnxruntime as ort
import os
from collections import Counter

FIQ_ONNX = "/content/qat_output/yolov8m_qat/weights/best_qat_fiq_packed.onnx"

OPT_ONNX = (
    "/content/qat_output/yolov8m_qat/weights/"
    "best_qat_fiq_ort_optimized.onnx"
)

print("=" * 80)
print("ONNX RUNTIME OPTIMIZED GRAPH ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# Create ORT session with maximum graph optimization
# and save the optimized graph.
# ------------------------------------------------------------

so = ort.SessionOptions()

so.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)

so.optimized_model_filepath = OPT_ONNX

print("\nCreating ONNX Runtime session...")

session = ort.InferenceSession(
    FIQ_ONNX,
    sess_options=so,
    providers=["CPUExecutionProvider"]
)

print("Session created successfully.")

# ------------------------------------------------------------
# Check optimized file
# ------------------------------------------------------------

if not os.path.isfile(OPT_ONNX):
    raise RuntimeError(
        "Optimized ONNX file was not created."
    )

print("\nOptimized graph saved:")
print(OPT_ONNX)

print(
    "\nOriginal size   : "
    f"{os.path.getsize(FIQ_ONNX)/(1024*1024):.2f} MB"
)

print(
    "Optimized size  : "
    f"{os.path.getsize(OPT_ONNX)/(1024*1024):.2f} MB"
)

# ------------------------------------------------------------
# Load optimized graph
# ------------------------------------------------------------

optimized = onnx.load(OPT_ONNX)

onnx.checker.check_model(optimized)

print("\nONNX checker: PASS")

# ------------------------------------------------------------
# Operator counts
# ------------------------------------------------------------

ops = Counter(
    node.op_type
    for node in optimized.graph.node
)

print("\n" + "=" * 80)
print("OPTIMIZED OPERATOR COUNTS")
print("=" * 80)

for op, count in ops.most_common():
    print(f"{op:<30} {count}")

# ------------------------------------------------------------
# Quantization-specific operators
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("QUANTIZATION OPERATORS")
print("=" * 80)

print(
    "QuantizeLinear    :",
    ops.get("QuantizeLinear", 0)
)

print(
    "DequantizeLinear  :",
    ops.get("DequantizeLinear", 0)
)

print(
    "QLinearConv       :",
    ops.get("QLinearConv", 0)
)

print(
    "ConvInteger       :",
    ops.get("ConvInteger", 0)
)

print(
    "MatMulInteger     :",
    ops.get("MatMulInteger", 0)
)

print(
    "Gather             :",
    ops.get("Gather", 0)
)

# ------------------------------------------------------------
# Initializer dtypes
# ------------------------------------------------------------

TYPE_MAP = {
    onnx.TensorProto.FLOAT: "FLOAT32",
    onnx.TensorProto.FLOAT16: "FLOAT16",
    onnx.TensorProto.INT8: "INT8",
    onnx.TensorProto.UINT8: "UINT8",
    onnx.TensorProto.INT32: "INT32",
    onnx.TensorProto.INT64: "INT64",
}

initializer_counts = Counter(
    TYPE_MAP.get(
        init.data_type,
        f"TYPE_{init.data_type}"
    )
    for init in optimized.graph.initializer
)

print("\n" + "=" * 80)
print("OPTIMIZED INITIALIZER TYPES")
print("=" * 80)

for dtype, count in initializer_counts.items():
    print(
        f"{dtype:<15}: {count}"
    )

# ------------------------------------------------------------
# Conv nodes after optimization
# ------------------------------------------------------------

conv_nodes = [
    node
    for node in optimized.graph.node
    if node.op_type == "Conv"
]

print("\n" + "=" * 80)
print("REMAINING CONV NODES")
print("=" * 80)

print(
    "Remaining Conv nodes:",
    len(conv_nodes)
)

for i, node in enumerate(conv_nodes[:10], 1):

    print(f"\nConv #{i}")
    print("Name:", node.name)

    print("Inputs:")
    for inp in node.input:
        print("  ", inp)

    print("Outputs:")
    for out in node.output:
        print("  ", out)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

ONNX RUNTIME OPTIMIZED GRAPH ANALYSIS

Creating ONNX Runtime session...
Session created successfully.

Optimized graph saved:
/content/qat_output/yolov8m_qat/weights/best_qat_fiq_ort_optimized.onnx

Original size   : 25.15 MB
Optimized size  : 25.16 MB

ONNX checker: PASS

OPTIMIZED OPERATOR COUNTS
DequantizeLinear               157
Conv                           84
QuickGelu                      77
QuantizeLinear                 75
Concat                         17
Add                            14
Split                          9
ReorderInput                   9
ReorderOutput                  9
Reshape                        8
MaxPool                        3
Resize                         2
Sub                            2
Transpose                      1
Softmax                        1
Div                            1
Mul                            1
Sigmoid                        1

QUANTIZATION OPERATORS
QuantizeLinear    : 75
DequantizeLinear  : 157
QLinearConv       : 0
ConvIn

In [18]:
from ultralytics import YOLO

FIQ_ONNX = "/content/qat_output/yolov8m_qat/weights/best_qat_fiq_packed.onnx"
DATA_YAML = "/content/pcb_qat/dataset/content/pcb-2/data.yaml"

print("=" * 80)
print("QAT-FIQ VALIDATION")
print("=" * 80)

model = YOLO(FIQ_ONNX)

metrics = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=640,
    batch=1,
    device="cpu",
    workers=2,
    verbose=True
)

print("\n" + "=" * 80)
print("QAT-FIQ RESULTS")
print("=" * 80)

print(f"mAP50      : {metrics.box.map50:.6f}")
print(f"mAP50-95   : {metrics.box.map:.6f}")
print(f"Precision  : {metrics.box.mp:.6f}")
print(f"Recall     : {metrics.box.mr:.6f}")

QAT-FIQ VALIDATION
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Loading /content/qat_output/yolov8m_qat/weights/best_qat_fiq_packed.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 889.1±158.1 MB/s, size: 58.6 KB)
val: Scanning /content/pcb_qat/dataset/content/pcb-2/valid/labels.cache... 798 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 798/798 196.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 798/798 1.0s/it 13:28
                   all        798       1588      0.979      0.986      0.989      0.623
          missing_hole        118        229       0.99      0.996      0.995      0.657
            mouse_bite        140        280      0.979      0.986      0.987      0.629
          open_circuit        133        255      0.973 

In [19]:
from ultralytics import YOLO
import os
import json


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# MODEL PATHS
# ------------------------------------------------------------

# Original FP32 model
MODEL_FP32 = "/content/best (1).pt"

# Previous PTQ INT8 model
MODEL_PTQ_INT8 = "/content/best (1)_int8.onnx"

# QAT-trained checkpoint
MODEL_QAT = (
    "/content/qat_output/yolov8m_qat/weights/"
    "QAT_best.pt"
)

# QAT -> packed INT8 model
MODEL_QAT_FIQ = (
    "/content/qat_output/yolov8m_qat/weights/"
    "best_qat_fiq_packed.onnx"
)

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------

DATA_YAML = (
    "/content/pcb_qat/dataset/content/pcb-2/data.yaml"
)

# IMPORTANT:
# ALL models use exactly the same validation split
EVALUATION_SPLIT = "val"

# ------------------------------------------------------------
# EVALUATION SETTINGS
# ------------------------------------------------------------

IMAGE_SIZE = 640
BATCH_SIZE = 1
DEVICE = "cpu"

# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = "/content/model_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# CHECK FILES
# ============================================================

print("=" * 90)
print("CHECKING FILES")
print("=" * 90)

files_to_check = {
    "FP32 model": MODEL_FP32,
    "PTQ INT8 model": MODEL_PTQ_INT8,
    "QAT model": MODEL_QAT,
    "QAT-FIQ model": MODEL_QAT_FIQ,
    "Dataset YAML": DATA_YAML
}

all_files_exist = True

for name, path in files_to_check.items():

    if os.path.isfile(path):

        print(f"[OK]      {name}")
        print(f"          {path}")

    else:

        print(f"[MISSING] {name}")
        print(f"          {path}")

        all_files_exist = False

print()

if not all_files_exist:

    raise FileNotFoundError(
        "\nOne or more required files are missing."
    )


# ============================================================
# MODEL SIZE
# ============================================================

def get_model_size_mb(model_path):

    return (
        os.path.getsize(model_path)
        / (1024 * 1024)
    )


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_path, model_name):

    print("\n\n")

    print("=" * 90)
    print(f"EVALUATING: {model_name}")
    print("=" * 90)

    print(f"Model : {model_path}")
    print(f"Split : {EVALUATION_SPLIT}")
    print(f"Size  : {IMAGE_SIZE}")
    print(f"Batch : {BATCH_SIZE}")
    print(f"Device: {DEVICE}")

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    model = YOLO(model_path)

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    results = model.val(
        data=DATA_YAML,
        split=EVALUATION_SPLIT,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        plots=True,
        verbose=True
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    precision = float(results.box.mp)
    recall = float(results.box.mr)

    map50 = float(results.box.map50)
    map5095 = float(results.box.map)

    # --------------------------------------------------------
    # F1 score
    # --------------------------------------------------------

    if precision + recall > 0:

        f1 = (
            2.0
            * precision
            * recall
            / (precision + recall)
        )

    else:

        f1 = 0.0

    # --------------------------------------------------------
    # Model size
    # --------------------------------------------------------

    model_size_mb = get_model_size_mb(model_path)

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    metrics = {

        "model": model_name,

        "model_path": model_path,

        "precision": precision,

        "recall": recall,

        "f1_score": f1,

        "mAP50": map50,

        "mAP50-95": map5095,

        "model_size_MB": model_size_mb,

        "image_size": IMAGE_SIZE,

        "batch_size": BATCH_SIZE,

        "device": DEVICE,

        "evaluation_split": EVALUATION_SPLIT
    }

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print("\n")
    print("-" * 60)
    print(f"{model_name} RESULTS")
    print("-" * 60)

    print(f"Precision   : {precision:.6f}")
    print(f"Recall      : {recall:.6f}")
    print(f"F1 Score    : {f1:.6f}")
    print(f"mAP50       : {map50:.6f}")
    print(f"mAP50-95    : {map5095:.6f}")
    print(f"Model Size  : {model_size_mb:.2f} MB")

    print("-" * 60)

    # --------------------------------------------------------
    # Save individual JSON
    # --------------------------------------------------------

    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{safe_name}_metrics.json"
    )

    with open(output_file, "w") as f:

        json.dump(
            metrics,
            f,
            indent=4
        )

    print("\nSaved:")
    print(output_file)

    return metrics


# ============================================================
# EVALUATE ALL FOUR MODELS
# ============================================================

fp32_metrics = evaluate_model(
    MODEL_FP32,
    "FP32"
)

ptq_metrics = evaluate_model(
    MODEL_PTQ_INT8,
    "PTQ INT8"
)

qat_metrics = evaluate_model(
    MODEL_QAT,
    "QAT"
)

qat_fiq_metrics = evaluate_model(
    MODEL_QAT_FIQ,
    "QAT-FIQ"
)


# ============================================================
# COMBINED RESULTS
# ============================================================

comparison = {

    "FP32": fp32_metrics,

    "PTQ_INT8": ptq_metrics,

    "QAT": qat_metrics,

    "QAT_FIQ": qat_fiq_metrics
}


comparison_file = os.path.join(
    OUTPUT_DIR,
    "four_model_comparison.json"
)

with open(comparison_file, "w") as f:

    json.dump(
        comparison,
        f,
        indent=4
    )


# ============================================================
# PERCENTAGE CHANGE
# ============================================================

def percentage_change(reference, value):

    if reference == 0:

        return 0.0

    return (
        (value - reference)
        / reference
    ) * 100.0


metrics_to_compare = [

    ("Precision", "precision"),

    ("Recall", "recall"),

    ("F1 Score", "f1_score"),

    ("mAP50", "mAP50"),

    ("mAP50-95", "mAP50-95"),

    ("Model Size (MB)", "model_size_MB")
]


# ============================================================
# CHANGE VS FP32
# ============================================================

change_vs_fp32 = {

    "PTQ_INT8": {},

    "QAT": {},

    "QAT_FIQ": {}
}


for label, key in metrics_to_compare:

    fp32_value = fp32_metrics[key]

    change_vs_fp32["PTQ_INT8"][label] = (
        percentage_change(
            fp32_value,
            ptq_metrics[key]
        )
    )

    change_vs_fp32["QAT"][label] = (
        percentage_change(
            fp32_value,
            qat_metrics[key]
        )
    )

    change_vs_fp32["QAT_FIQ"][label] = (
        percentage_change(
            fp32_value,
            qat_fiq_metrics[key]
        )
    )


# ============================================================
# QAT-FIQ VS PTQ
# ============================================================

qat_fiq_vs_ptq = {}

for label, key in metrics_to_compare:

    qat_fiq_vs_ptq[label] = percentage_change(
        ptq_metrics[key],
        qat_fiq_metrics[key]
    )


# ============================================================
# QAT-FIQ VS QAT
# ============================================================

qat_fiq_vs_qat = {}

for label, key in metrics_to_compare:

    qat_fiq_vs_qat[label] = percentage_change(
        qat_metrics[key],
        qat_fiq_metrics[key]
    )


# ============================================================
# FINAL TABLE
# ============================================================

print("\n\n")

print("=" * 110)
print("                         FOUR-MODEL COMPARISON")
print("=" * 110)

print(
    f"{'Metric':<22}"
    f"{'FP32':>15}"
    f"{'PTQ INT8':>15}"
    f"{'QAT':>15}"
    f"{'QAT-FIQ':>15}"
)

print("-" * 110)

for label, key in metrics_to_compare:

    print(
        f"{label:<22}"
        f"{fp32_metrics[key]:>15.4f}"
        f"{ptq_metrics[key]:>15.4f}"
        f"{qat_metrics[key]:>15.4f}"
        f"{qat_fiq_metrics[key]:>15.4f}"
    )

print("=" * 110)


# ============================================================
# CHANGE VS FP32
# ============================================================

print("\n")

print("=" * 110)
print("                       CHANGE RELATIVE TO FP32")
print("=" * 110)

print(
    f"{'Metric':<22}"
    f"{'PTQ vs FP32':>20}"
    f"{'QAT vs FP32':>20}"
    f"{'QAT-FIQ vs FP32':>20}"
)

print("-" * 110)

for label, key in metrics_to_compare:

    print(
        f"{label:<22}"
        f"{change_vs_fp32['PTQ_INT8'][label]:>19.2f}%"
        f"{change_vs_fp32['QAT'][label]:>19.2f}%"
        f"{change_vs_fp32['QAT_FIQ'][label]:>19.2f}%"
    )

print("=" * 110)


# ============================================================
# QAT-FIQ VS PTQ
# ============================================================

print("\n")

print("=" * 110)
print("                         QAT-FIQ VS PTQ")
print("=" * 110)

print(
    f"{'Metric':<22}"
    f"{'PTQ INT8':>15}"
    f"{'QAT-FIQ':>15}"
    f"{'Change':>20}"
)

print("-" * 110)

for label, key in metrics_to_compare:

    print(
        f"{label:<22}"
        f"{ptq_metrics[key]:>15.4f}"
        f"{qat_fiq_metrics[key]:>15.4f}"
        f"{qat_fiq_vs_ptq[label]:>19.2f}%"
    )

print("=" * 110)


# ============================================================
# QAT-FIQ VS QAT
# ============================================================

print("\n")

print("=" * 110)
print("                         QAT-FIQ VS QAT")
print("=" * 110)

print(
    f"{'Metric':<22}"
    f"{'QAT':>15}"
    f"{'QAT-FIQ':>15}"
    f"{'Change':>20}"
)

print("-" * 110)

for label, key in metrics_to_compare:

    print(
        f"{label:<22}"
        f"{qat_metrics[key]:>15.4f}"
        f"{qat_fiq_metrics[key]:>15.4f}"
        f"{qat_fiq_vs_qat[label]:>19.2f}%"
    )

print("=" * 110)


# ============================================================
# INTERPRETATION
# ============================================================

print("\n")

print("=" * 110)
print("                            INTERPRETATION")
print("=" * 110)

# mAP50-95
if qat_fiq_metrics["mAP50-95"] > ptq_metrics["mAP50-95"]:

    print(
        "✓ QAT-FIQ has higher mAP50-95 than PTQ INT8."
    )

elif qat_fiq_metrics["mAP50-95"] < ptq_metrics["mAP50-95"]:

    print(
        "✗ QAT-FIQ has lower mAP50-95 than PTQ INT8."
    )

else:

    print(
        "= QAT-FIQ and PTQ INT8 have identical mAP50-95."
    )


# mAP50
if qat_fiq_metrics["mAP50"] > ptq_metrics["mAP50"]:

    print(
        "✓ QAT-FIQ has higher mAP50 than PTQ INT8."
    )

elif qat_fiq_metrics["mAP50"] < ptq_metrics["mAP50"]:

    print(
        "✗ QAT-FIQ has lower mAP50 than PTQ INT8."
    )

else:

    print(
        "= QAT-FIQ and PTQ INT8 have identical mAP50."
    )


# QAT -> QAT-FIQ preservation
map_loss = (
    qat_metrics["mAP50-95"]
    - qat_fiq_metrics["mAP50-95"]
)

print(
    f"\nQAT → QAT-FIQ mAP50-95 difference: "
    f"{map_loss:.6f}"
)

# Size
print(
    f"\nModel sizes:"
)

print(
    f"  FP32    : "
    f"{fp32_metrics['model_size_MB']:.2f} MB"
)

print(
    f"  PTQ INT8: "
    f"{ptq_metrics['model_size_MB']:.2f} MB"
)

print(
    f"  QAT     : "
    f"{qat_metrics['model_size_MB']:.2f} MB"
)

print(
    f"  QAT-FIQ : "
    f"{qat_fiq_metrics['model_size_MB']:.2f} MB"
)

print("=" * 110)


# ============================================================
# SAVE COMPLETE RESULTS
# ============================================================

complete_results = {

    "evaluation_configuration": {

        "data_yaml": DATA_YAML,

        "split": EVALUATION_SPLIT,

        "image_size": IMAGE_SIZE,

        "batch_size": BATCH_SIZE,

        "device": DEVICE
    },

    "models": {

        "FP32": fp32_metrics,

        "PTQ_INT8": ptq_metrics,

        "QAT": qat_metrics,

        "QAT_FIQ": qat_fiq_metrics
    },

    "change_vs_FP32": change_vs_fp32,

    "QAT_FIQ_vs_PTQ": qat_fiq_vs_ptq,

    "QAT_FIQ_vs_QAT": qat_fiq_vs_qat
}


complete_file = os.path.join(
    OUTPUT_DIR,
    "complete_four_model_results.json"
)

with open(complete_file, "w") as f:

    json.dump(
        complete_results,
        f,
        indent=4
    )


# ============================================================
# DONE
# ============================================================

print("\n")

print("=" * 90)
print("EVALUATION COMPLETE")
print("=" * 90)

print("\nResults saved in:")
print(OUTPUT_DIR)

print("\nGenerated files:")

print("  FP32_metrics.json")
print("  ptq_int8_metrics.json")
print("  qat_metrics.json")
print("  qat_fiq_metrics.json")
print("  four_model_comparison.json")
print("  complete_four_model_results.json")

print("\nAll four models were evaluated on:")
print(f"  Split      : {EVALUATION_SPLIT}")
print(f"  Image size : {IMAGE_SIZE}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Device     : {DEVICE}")

print("=" * 90)

CHECKING FILES
[OK]      FP32 model
          /content/best (1).pt
[OK]      PTQ INT8 model
          /content/best (1)_int8.onnx
[OK]      QAT model
          /content/qat_output/yolov8m_qat/weights/QAT_best.pt
[OK]      QAT-FIQ model
          /content/qat_output/yolov8m_qat/weights/best_qat_fiq_packed.onnx
[OK]      Dataset YAML
          /content/pcb_qat/dataset/content/pcb-2/data.yaml




EVALUATING: FP32
Model : /content/best (1).pt
Split : val
Size  : 640
Batch : 1
Device: cpu
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1153.8±234.9 MB/s, size: 64.2 KB)
val: Scanning /content/pcb_qat/dataset/content/pcb-2/valid/labels.cache... 798 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 798/798 159.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━

/usr/local/lib/python3.13/dist-packages/modelopt/torch/quantization/nn/modules/quant_module.py:111: UserWarning: Could not identify the device for TensorQuantizer states of model.9.m. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 261 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1069.5±269.7 MB/s, size: 57.3 KB)
val: Scanning /content/pcb_qat/dataset/content/pcb-2/valid/labels.cache... 798 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 798/798 196.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 798/798 1.8s/it 23:30
                   all        798       1588       0.98      0.991       0.99      0.625
          missing_hole        118        229      0.997      0.996      0.995       0.66
            mouse_bite        140        280      0.978      0.989      0.988      0.631
          open_circuit        133        255      0.984      0.994      0.991       0.61
                 short        156        324      0.978      0.977      0.988      0.635
                